# 📈 Prediksi Interval Harga Saham LQ45
## Menggunakan Long Short-Term Memory (LSTM) dan Quantile Regression
---
**Penelitian Ilmiah**

| | |
|---|---|
| **Dataset** | IDX Stock Summary 2020–2024 |
| **Model** | LSTM + Quantile Regression |
| **Target** | Prediksi interval harga (batas bawah Q10, median Q50, batas atas Q90) |
| **Evaluasi** | RMSE & Calibration Error (CE) |

---


## 1. Instalasi Library & Import

In [ ]:
# Install library tambahan jika diperlukan
!pip install tensorflow scikit-learn matplotlib seaborn pandas numpy -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, LSTM, Dense, Dropout, BatchNormalization)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
import tensorflow.keras.backend as K

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("✅ TensorFlow version:", tf.__version__)
print("✅ GPU tersedia:", len(tf.config.list_physical_devices('GPU')) > 0)
print("✅ Semua library berhasil diimport!")

---
## 2. Load & Preprocessing Dataset

### 2.1 Upload Dataset
Upload file **IDX Stock Summary 2020-2024.csv** dari dataset Anda.


In [ ]:
# ── Load dataset dari Google Drive ───────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

file_path = '/content/drive/My Drive/PI/Dataset/IDX Stock Summary 2020-2024.csv'
df_raw = pd.read_csv(file_path, sep=';')
print(f"✅ Dataset berhasil diload: {df_raw.shape[0]:,} baris, {df_raw.shape[1]} kolom")

### 2.2 Parsing & Cleaning Data

In [ ]:
# ── Parse tanggal: ganti 'Agt' (Indonesia) → 'Aug' (Inggris) lalu parse ────────
df_raw['Date'] = df_raw['Last Trading Date'].str.replace('Agt', 'Aug', regex=False)
df_raw['Date'] = pd.to_datetime(df_raw['Date'], dayfirst=True)

# ── Daftar 38 saham LQ45 dengan data ≥ 80% ───────────────────────────────────
LQ45_VALID = [
    'ADRO','AKRA','AMRT','ANTM','ASII','BBCA','BBNI','BBRI','BBTN','BMRI',
    'BRPT','BUMI','CPIN','DEWA','EMTK','ESSA','EXCL','HRTA','ICBP','INCO',
    'INDF','INKP','ISAT','ITMG','JPFA','KLBF','MAPI','MDKA','MEDC','PGAS',
    'PTBA','SCMA','SMGR','TLKM','TOWR','UNTR','UNVR','WIFI'
]

# ── Filter & cleaning ─────────────────────────────────────────────────────────
df = df_raw[df_raw['Stock Code'].isin(LQ45_VALID)].copy()
df = df[df['Volume'] > 0].copy()
df.rename(columns={
    'Stock Code': 'Stock_Code',
    'Company Name': 'Company_Name',
    'Open Price': 'Open',
}, inplace=True)

# Fix Open = 0 → ganti dengan Previous
df.loc[df['Open'] == 0, 'Open'] = df.loc[df['Open'] == 0, 'Previous']

df = df[['Date','Stock_Code','Company_Name','Open','High','Low','Close',
         'Volume','Bid','Offer']].copy()
df.sort_values(['Stock_Code','Date'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"✅ Total saham LQ45 valid : {df['Stock_Code'].nunique()}")
print(f"✅ Total baris            : {len(df):,}")
print(f"✅ Rentang tanggal        : {df['Date'].min().date()} — {df['Date'].max().date()}")
print(f"✅ Missing values         : {df.isnull().sum().sum()}")
df.head()

---
## 3. Eksplorasi & Visualisasi Dataset (EDA)


### 3.1 Distribusi Jumlah Data per Saham

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
count_per_stock = df.groupby('Stock_Code')['Date'].count().sort_values(ascending=True)
colors = ['#2196F3' if v == count_per_stock.max() else
          '#FF5722' if v < 500 else '#4CAF50'
          for v in count_per_stock.values]
bars = ax.barh(count_per_stock.index, count_per_stock.values, color=colors, edgecolor='white', height=0.7)
ax.axvline(x=500, color='red', linestyle='--', linewidth=1.2, label='Batas minimum (500 hari)')
ax.set_xlabel('Jumlah Hari Trading', fontsize=12)
ax.set_title('Jumlah Hari Trading per Saham LQ45 dalam Dataset', fontsize=14, fontweight='bold')
ax.legend()
for bar, val in zip(bars, count_per_stock.values):
    ax.text(val + 8, bar.get_y() + bar.get_height()/2, str(val), va='center', fontsize=8)
plt.tight_layout()
plt.savefig('viz_01_jumlah_hari_per_saham.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Visualisasi 1: Jumlah hari trading per saham")

### 3.2 Tren Harga Close Saham-Saham Utama LQ45 (2020–2024)

In [ ]:
SAHAM_HIGHLIGHT = ['BBCA','BBRI','TLKM','ASII','ANTM','ADRO']
fig, axes = plt.subplots(3, 2, figsize=(16, 12))
axes = axes.flatten()
colors_line = ['#1565C0','#C62828','#2E7D32','#E65100','#6A1B9A','#00838F']

for i, (kode, col) in enumerate(zip(SAHAM_HIGHLIGHT, colors_line)):
    sub = df[df['Stock_Code'] == kode].copy()
    if sub.empty:
        print(f'⚠️ {kode} tidak ditemukan, dilewati')
        continue
    ax = axes[i]
    ax.plot(sub['Date'], sub['Close'], color=col, linewidth=1.2, label=kode)
    ax.fill_between(sub['Date'], sub['Low'], sub['High'], alpha=0.15, color=col)
    ax.set_title(f'{kode} — {sub["Company_Name"].iloc[0]}', fontsize=11, fontweight='bold')
    ax.set_ylabel('Harga (Rp)', fontsize=9)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'Rp {x:,.0f}'))
    ax.grid(alpha=0.3)
    # Tandai COVID crash
    ax.axvline(pd.Timestamp('2020-03-01'), color='red', linestyle=':', alpha=0.6, linewidth=1)

plt.suptitle('Tren Harga Close Saham LQ45 (2020-2024)\n(Area = rentang High-Low harian | Garis merah = awal COVID-19)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('viz_02_tren_harga_saham_utama.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Visualisasi 2: Tren harga saham utama LQ45")

### 3.3 Distribusi Return Harian per Saham

In [ ]:
# Hitung return harian
df['Return'] = df.groupby('Stock_Code')['Close'].pct_change()
df.dropna(subset=['Return'], inplace=True)

fig, axes = plt.subplots(6, 7, figsize=(20, 16))
axes = axes.flatten()
saham_list = sorted(df['Stock_Code'].unique())

for i, kode in enumerate(saham_list):
    sub = df[df['Stock_Code'] == kode]['Return']
    axes[i].hist(sub, bins=50, color='#1565C0', alpha=0.75, edgecolor='white', linewidth=0.3)
    axes[i].axvline(0, color='red', linewidth=1, linestyle='--')
    axes[i].set_title(kode, fontsize=9, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('')
    axes[i].tick_params(labelsize=7)

# Sembunyikan subplot kosong
for j in range(len(saham_list), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribusi Return Harian Saham LQ45\n(Garis merah = return = 0)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('viz_03_distribusi_return.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Visualisasi 3: Distribusi return harian semua saham LQ45")

### 3.4 Heatmap Korelasi Harga Close antar Saham Utama

In [ ]:
pivot = df.pivot_table(index='Date', columns='Stock_Code', values='Close')
cols_available = [c for c in SAHAM_HIGHLIGHT + ['BMRI','INDF','KLBF','PTBA'] if c in pivot.columns]
corr = pivot[cols_available].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=-1, vmax=1, ax=ax, linewidths=0.5,
            annot_kws={'size': 10})
ax.set_title('Heatmap Korelasi Harga Close antar Saham LQ45 Pilihan',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('viz_04_heatmap_korelasi.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Visualisasi 4: Heatmap korelasi harga close")

### 3.5 Volatilitas (Std. Dev. Rolling 20 Hari) Saham Utama

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
colors_vol = ['#1565C0','#C62828','#2E7D32','#E65100','#6A1B9A','#00838F']

for kode, col in zip(SAHAM_HIGHLIGHT, colors_vol):
    sub = df[df['Stock_Code'] == kode].copy()
    if sub.empty:
        continue
    sub['Vol20'] = sub['Return'].rolling(20).std() * np.sqrt(252) * 100  # annualized %
    ax.plot(sub['Date'], sub['Vol20'], label=kode, color=col, linewidth=1.2, alpha=0.85)

ax.set_title('Volatilitas Tahunan Rolling 20 Hari (%) Saham LQ45 Utama', fontsize=13, fontweight='bold')
ax.set_ylabel('Volatilitas Annualized (%)', fontsize=11)
ax.set_xlabel('Tanggal', fontsize=11)
ax.axvline(pd.Timestamp('2020-03-01'), color='gray', linestyle='--', linewidth=1, label='COVID-19')
ax.legend(loc='upper right', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('viz_05_volatilitas_rolling.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Visualisasi 5: Volatilitas rolling 20 hari")

### 3.6 Statistik Deskriptif Dataset

In [ ]:
stats = df.groupby('Stock_Code')['Close'].agg(['min','max','mean','std']).round(0)
stats.columns = ['Harga Min','Harga Max','Harga Rata-rata','Std Dev']
stats['Return Rata-rata (%)'] = (df.groupby('Stock_Code')['Return'].mean() * 100).round(4)
stats['Volatilitas (%)'] = (df.groupby('Stock_Code')['Return'].std() * np.sqrt(252) * 100).round(2)

print("=" * 75)
print("STATISTIK DESKRIPTIF SAHAM LQ45 (2020–2024)")
print("=" * 75)
print(stats.to_string())
print("=" * 75)

---
## 4. Persiapan Fitur OHLCV

Sesuai dengan judul penelitian, input model hanya menggunakan **5 fitur OHLCV**:

| Fitur | Keterangan |
|---|---|
| **Open** | Harga pembukaan |
| **High** | Harga tertinggi dalam sehari |
| **Low** | Harga terendah dalam sehari |
| **Close** | Harga penutupan |
| **Volume** | Total lot yang diperdagangkan |

**Target prediksi:** `Close` hari berikutnya (t+1)


In [ ]:
# ── Feature Engineering berbasis RETURN (pola perubahan harga) ───────────────
# Pendekatan ini membuat model belajar POLA NAIK/TURUN, bukan harga absolut.
# Sehingga hasil prediksi tetap valid meski harga sekarang berbeda dari data training.

def build_features(df):
    df = df.copy()
    df.sort_values(['Stock_Code','Date'], inplace=True)
    grp = df.groupby('Stock_Code')

    # Return harian: perubahan relatif harga Close (%)
    df['Return_O'] = grp['Open'].pct_change()     # return Open
    df['Return_H'] = grp['High'].pct_change()     # return High
    df['Return_L'] = grp['Low'].pct_change()      # return Low
    df['Return_C'] = grp['Close'].pct_change()    # return Close (utama)

    # Volume dinormalisasi relatif (z-score rolling per saham)
    df['Volume_Rel'] = grp['Volume'].transform(
        lambda x: (x - x.rolling(20).mean()) / (x.rolling(20).std() + 1e-8)
    )

    # Target: Return Close hari berikutnya (bukan harga absolut!)
    # Model memprediksi: besok naik/turun berapa persen dari Close hari ini
    df['Target_Return'] = grp['Return_C'].shift(-1)

    df.dropna(inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df

df_feat = build_features(df)

# Fitur input: return OHLCV (pola perubahan), bukan harga absolut
FEATURE_COLS = ['Return_O', 'Return_H', 'Return_L', 'Return_C', 'Volume_Rel']
TARGET_COL   = 'Target_Return'

print(f'✅ Total fitur   : {len(FEATURE_COLS)} (Return-based OHLCV)')
print(f'✅ Total baris   : {len(df_feat):,}')
print(f'✅ Total saham   : {df_feat["Stock_Code"].nunique()}')
print('\nFitur yang digunakan:')
for i, f in enumerate(FEATURE_COLS, 1):
    print(f'   {i}. {f}')
print('\nTarget : Target_Return (return Close hari berikutnya)')
df_feat[FEATURE_COLS + ['Target_Return','Close']].head()


---
## 5. Persiapan Data Training

### 5.1 Normalisasi & Sliding Window


In [ ]:
WINDOW_SIZE = 30
TEST_RATIO  = 0.2

def prepare_sequences(df_stock, feature_cols, target_col, window=30):
    """
    Buat sekuens LSTM dari data return.
    Simpan juga Close asli untuk konversi hasil prediksi kembali ke harga.
    """
    # Fitur: return (tidak perlu scaler karena sudah dalam skala %)
    X_vals = df_stock[feature_cols].values
    y_vals = df_stock[target_col].values
    close_vals = df_stock['Close'].values

    X_seq, y_seq, close_seq = [], [], []
    for i in range(window, len(X_vals)):
        X_seq.append(X_vals[i-window:i])
        y_seq.append(y_vals[i])
        close_seq.append(close_vals[i-1])  # Close hari ini (sebelum prediksi)

    return np.array(X_seq), np.array(y_seq), np.array(close_seq)

# Buat dataset gabungan semua saham
X_all, y_all, close_all = [], [], []
dates_all, stocks_all   = [], []

saham_list = sorted(df_feat['Stock_Code'].unique())

for kode in saham_list:
    sub = df_feat[df_feat['Stock_Code'] == kode].reset_index(drop=True)
    if len(sub) < WINDOW_SIZE + 50:
        print(f'⚠️  {kode}: data terlalu sedikit, skip')
        continue
    X_seq, y_seq, close_seq = prepare_sequences(sub, FEATURE_COLS, TARGET_COL, WINDOW_SIZE)
    dates_seq = sub['Date'].iloc[WINDOW_SIZE:].values

    X_all.append(X_seq)
    y_all.append(y_seq)
    close_all.append(close_seq)
    dates_all.append(dates_seq)
    stocks_all.extend([kode] * len(X_seq))

X_all     = np.concatenate(X_all,     axis=0)
y_all     = np.concatenate(y_all,     axis=0)
close_all = np.concatenate(close_all, axis=0)
dates_all = np.concatenate(dates_all, axis=0)
stocks_all = np.array(stocks_all)

# Split train/test (time-based)
n_total = len(X_all)
n_train = int(n_total * (1 - TEST_RATIO))

X_train,     X_test     = X_all[:n_train],     X_all[n_train:]
y_train,     y_test     = y_all[:n_train],     y_all[n_train:]
close_train, close_test = close_all[:n_train], close_all[n_train:]
dates_test  = dates_all[n_train:]
stocks_test = stocks_all[n_train:]

print(f'✅ Total sampel  : {n_total:,}')
print(f'✅ Training set  : {len(X_train):,} sampel (80%)')
print(f'✅ Test set      : {len(X_test):,} sampel (20%)')
print(f'✅ Input shape   : {X_train.shape}  → (samples, window={WINDOW_SIZE}, features={len(FEATURE_COLS)})')


---
## 6. Membangun Model LSTM + Quantile Regression

### 6.1 Pinball Loss Function
*Pinball Loss* (atau *Quantile Loss*) adalah fungsi loss yang digunakan untuk melatih
model Quantile Regression. Untuk kuantil τ:

$$L_\tau(y, \hat{y}) = \max(\tau(y - \hat{y}),\ (\tau-1)(y - \hat{y}))$$


In [ ]:
def pinball_loss(quantile):
    """
    Smoothed Pinball Loss untuk Quantile Regression.
    quantile: float antara 0 dan 1 (misal 0.1, 0.5, 0.9)
    """
    def loss(y_true, y_pred):
        error = y_true - y_pred
        return K.mean(K.maximum(quantile * error, (quantile - 1) * error))
    loss.__name__ = f'pinball_q{int(quantile*100)}'
    return loss

print("✅ Pinball Loss function siap")
print(f"   Kuantil yang akan dilatih: Q10 (batas bawah), Q50 (median), Q90 (batas atas)")

### 6.2 Arsitektur Model LSTM

In [ ]:
def build_lstm_quantile(input_shape, quantile, units_1=128, units_2=64, dropout=0.2):
    """
    Membangun model LSTM + Quantile Regression.

    Arsitektur:
    Input → LSTM(128) → Dropout → LSTM(64) → Dropout → BatchNorm → Dense(32) → Dense(1)

    Setiap kuantil (Q10, Q50, Q90) dilatih sebagai model terpisah
    dengan fungsi loss Pinball yang berbeda.
    """
    inputs = Input(shape=input_shape, name='input_ohlcv')

    # LSTM Layer 1
    x = LSTM(units_1, return_sequences=True, name='lstm_1')(inputs)
    x = Dropout(dropout, name='dropout_1')(x)

    # LSTM Layer 2
    x = LSTM(units_2, return_sequences=False, name='lstm_2')(x)
    x = Dropout(dropout, name='dropout_2')(x)

    # Batch Normalization
    x = BatchNormalization(name='batch_norm')(x)

    # Dense layers
    x = Dense(32, activation='relu', name='dense_1')(x)
    output = Dense(1, activation='linear', name='output')(x)

    model = Model(inputs=inputs, outputs=output,
                  name=f'LSTM_QR_Q{int(quantile*100)}')
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss=pinball_loss(quantile)
    )
    return model

# Preview arsitektur dengan Q50
model_preview = build_lstm_quantile(
    input_shape=(WINDOW_SIZE, len(FEATURE_COLS)),
    quantile=0.5
)
model_preview.summary()

### 6.3 Training Model untuk Q10, Q50, Q90

In [ ]:
QUANTILES   = [0.1, 0.5, 0.9]
EPOCHS      = 50
BATCH_SIZE  = 256

callbacks = [
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=0),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=0)
]

models    = {}
histories = {}

print("=" * 60)
print("TRAINING LSTM + QUANTILE REGRESSION")
print("=" * 60)

for q in QUANTILES:
    print(f"\n🔄 Training model Q{int(q*100)} (kuantil {q}) ...")
    model = build_lstm_quantile(
        input_shape=(WINDOW_SIZE, len(FEATURE_COLS)),
        quantile=q,
        units_1=128, units_2=64, dropout=0.2
    )
    history = model.fit(
        X_train, y_train,
        validation_split=0.1,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=0
    )
    models[q]    = model
    histories[q] = history
    best_epoch   = np.argmin(history.history['val_loss']) + 1
    best_loss    = min(history.history['val_loss'])
    print(f"   ✅ Selesai! Best epoch: {best_epoch} | Val Loss: {best_loss:.6f}")

print("\n✅ Semua model selesai dilatih!")

### 6.4 Training Loss per Kuantil

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
colors_q = {0.1: '#E53935', 0.5: '#1565C0', 0.9: '#2E7D32'}
labels_q = {0.1: 'Q10 (Batas Bawah)', 0.5: 'Q50 (Median)', 0.9: 'Q90 (Batas Atas)'}

for i, q in enumerate(QUANTILES):
    h = histories[q].history
    ax = axes[i]
    ax.plot(h['loss'], label='Train Loss', color=colors_q[q], linewidth=2)
    ax.plot(h['val_loss'], label='Val Loss', color=colors_q[q], linewidth=2, linestyle='--')
    ax.set_title(f'{labels_q[q]}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Pinball Loss')
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('Training & Validation Loss per Kuantil LSTM-QR', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('viz_06_training_loss.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Visualisasi 6: Training loss per kuantil")

---
## 7. Prediksi & Evaluasi Model

### 7.1 Prediksi Interval pada Test Set


In [ ]:
# ── Prediksi Return pada test set ────────────────────────────────────────────
y_pred_q10_ret = models[0.1].predict(X_test, verbose=0).flatten()
y_pred_q50_ret = models[0.5].predict(X_test, verbose=0).flatten()
y_pred_q90_ret = models[0.9].predict(X_test, verbose=0).flatten()

# ── Konversi Return → Harga Absolut ──────────────────────────────────────────
# harga_prediksi = close_hari_ini × (1 + return_prediksi)
y_true_price = close_test * (1 + y_test)          # harga aktual besok
y_q10_price  = close_test * (1 + y_pred_q10_ret)  # batas bawah prediksi
y_q50_price  = close_test * (1 + y_pred_q50_ret)  # median prediksi
y_q90_price  = close_test * (1 + y_pred_q90_ret)  # batas atas prediksi

# Alias untuk kompatibilitas kode evaluasi di bawah
y_true_inv = y_true_price
y_q10_inv  = y_q10_price
y_q50_inv  = y_q50_price
y_q90_inv  = y_q90_price

print(f'✅ Prediksi selesai pada {len(y_test):,} sampel test')
print(f'\nContoh 5 prediksi pertama (dalam harga Rp):')
print(f'{"Aktual":>12} {"Q10 (Bawah)":>14} {"Q50 (Median)":>14} {"Q90 (Atas)":>12}')
print('-' * 55)
for i in range(5):
    print(f'{y_true_inv[i]:>12,.0f} {y_q10_inv[i]:>14,.0f} {y_q50_inv[i]:>14,.0f} {y_q90_inv[i]:>12,.0f}')


### 7.2 Evaluasi RMSE (Root Mean Square Error)

$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

RMSE dihitung pada prediksi **Q50 (median)** sebagai nilai prediksi titik utama.


In [ ]:
from sklearn.metrics import mean_squared_error

def calc_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

rmse_overall = calc_rmse(y_true_inv, y_q50_inv)

print('=' * 55)
print('HASIL EVALUASI RMSE')
print('=' * 55)
print(f'RMSE Keseluruhan (Q50 vs Aktual) : {rmse_overall:,.2f}')
print('=' * 55)

# RMSE per saham
rmse_per_saham = {}
unique_stocks_test = np.unique(stocks_test)

for kode in unique_stocks_test:
    idx = stocks_test == kode
    if idx.sum() < 10: continue
    y_t = y_true_inv[idx]
    y_p = y_q50_inv[idx]
    rmse_per_saham[kode] = calc_rmse(y_t, y_p)

rmse_df = pd.DataFrame.from_dict(rmse_per_saham, orient='index', columns=['RMSE'])
rmse_df.index.name = 'Stock_Code'
rmse_df = rmse_df.sort_values('RMSE')

print(f'\nRMSE per Saham (diurutkan terbaik ke terburuk):')
print('-' * 35)
print(rmse_df.to_string())
print('-' * 35)
print(f'RMSE Terbaik   : {rmse_df["RMSE"].min():,.2f} ({rmse_df["RMSE"].idxmin()})')
print(f'RMSE Terburuk  : {rmse_df["RMSE"].max():,.2f} ({rmse_df["RMSE"].idxmax()})')
print(f'RMSE Rata-rata : {rmse_df["RMSE"].mean():,.2f}')


### 7.3 Visualisasi RMSE per Saham

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
rmse_sorted = rmse_df.sort_values('RMSE', ascending=True)
colors_rmse = ['#4CAF50' if v < rmse_df['RMSE'].quantile(0.33) else
               '#FFC107' if v < rmse_df['RMSE'].quantile(0.66) else
               '#F44336' for v in rmse_sorted['RMSE']]

bars = ax.barh(rmse_sorted.index, rmse_sorted['RMSE'], color=colors_rmse, edgecolor='white')
ax.axvline(rmse_df['RMSE'].mean(), color='navy', linestyle='--', linewidth=1.5, label=f'Rata-rata RMSE = {rmse_df["RMSE"].mean():,.0f}')
ax.set_xlabel('RMSE (Rp)', fontsize=12)
ax.set_title('RMSE per Saham LQ45 — Model LSTM + Quantile Regression (Q50)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)

# Patch legend
p1 = mpatches.Patch(color='#4CAF50', label='RMSE Rendah (Baik)')
p2 = mpatches.Patch(color='#FFC107', label='RMSE Sedang')
p3 = mpatches.Patch(color='#F44336', label='RMSE Tinggi')
ax.legend(handles=[p1, p2, p3], loc='lower right', fontsize=9)

for bar, val in zip(bars, rmse_sorted['RMSE']):
    ax.text(val + 5, bar.get_y() + bar.get_height()/2,
            f'{val:,.0f}', va='center', fontsize=8)
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('viz_07_rmse_per_saham.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Visualisasi 7: RMSE per saham")

### 7.4 Evaluasi Calibration Error (CE)

**Calibration Error** mengukur seberapa baik interval prediksi terkalibrasi:

$$CE = |\text{Coverage Empiris} - \tau|$$

Untuk interval 80% (Q10–Q90), idealnya **80% data aktual** jatuh di dalam interval prediksi.
Semakin kecil CE, semakin terkalibrasi model.


In [ ]:
def calibration_error(y_true, y_lower, y_upper, nominal_coverage=0.80):
    in_interval = (y_true >= y_lower) & (y_true <= y_upper)
    empirical_coverage = in_interval.mean()
    ce = abs(empirical_coverage - nominal_coverage)
    return ce, empirical_coverage

ce_overall, cov_overall = calibration_error(y_true_inv, y_q10_inv, y_q90_inv, 0.80)

print('=' * 60)
print('HASIL EVALUASI CALIBRATION ERROR (CE)')
print('=' * 60)
print(f'Level kepercayaan nominal (Q10-Q90) : 80.00%')
print(f'Coverage empiris aktual             : {cov_overall*100:.2f}%')
print(f'Calibration Error (CE)              : {ce_overall*100:.4f}%')
print('-' * 60)
if ce_overall < 0.05:
    print('✅ Model TERKALIBRASI SANGAT BAIK (CE < 5%)')
elif ce_overall < 0.10:
    print('✅ Model TERKALIBRASI BAIK (CE < 10%)')
else:
    print('⚠️  Model kurang terkalibrasi (CE >= 10%)')
print('=' * 60)

# CE per saham
ce_per_saham = {}
for kode in unique_stocks_test:
    idx = stocks_test == kode
    if idx.sum() < 10: continue
    ce, cov = calibration_error(y_true_inv[idx], y_q10_inv[idx], y_q90_inv[idx], 0.80)
    ce_per_saham[kode] = {'CE (%)': round(ce*100, 4), 'Coverage (%)': round(cov*100, 2)}

ce_df = pd.DataFrame(ce_per_saham).T
ce_df.index.name = 'Stock_Code'
ce_df = ce_df.sort_values('CE (%)')
print(f'\nCalibration Error per Saham:')
print(ce_df.to_string())


### 7.5 Visualisasi Calibration Error per Saham

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot CE per saham
ax1 = axes[0]
colors_ce = ['#4CAF50' if v < 5 else '#FFC107' if v < 10 else '#F44336'
             for v in ce_df['CE (%)']]
ce_sorted = ce_df.sort_values('CE (%)')
bars = ax1.barh(ce_sorted.index, ce_sorted['CE (%)'], color=colors_ce, edgecolor='white')
ax1.axvline(5, color='green', linestyle='--', linewidth=1.5, label='CE = 5% (sangat baik)')
ax1.axvline(10, color='red', linestyle='--', linewidth=1.5, label='CE = 10% (batas)')
ax1.set_xlabel('Calibration Error (%)', fontsize=11)
ax1.set_title('Calibration Error per Saham LQ45', fontsize=12, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3, axis='x')

# Plot Coverage vs Nominal
ax2 = axes[1]
coverage_sorted = ce_df.sort_values('Coverage (%)')
colors_cov = ['#4CAF50' if abs(v - 80) < 5 else '#FFC107' if abs(v - 80) < 10 else '#F44336'
              for v in coverage_sorted['Coverage (%)']]
ax2.barh(coverage_sorted.index, coverage_sorted['Coverage (%)'], color=colors_cov, edgecolor='white')
ax2.axvline(80, color='navy', linestyle='--', linewidth=2, label='Target Coverage 80%')
ax2.set_xlabel('Empirical Coverage (%)', fontsize=11)
ax2.set_title('Empirical Coverage per Saham LQ45\n(Target: 80%)', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3, axis='x')

plt.suptitle('Evaluasi Kalibrasi Model LSTM + Quantile Regression', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('viz_08_calibration_error.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Visualisasi 8: Calibration Error per saham")

---
## 8. Visualisasi Prediksi Interval Harga

### 8.1 Prediksi Interval Saham Pilihan


In [ ]:
def plot_prediction_interval(kode, n_days=60):
    idx = stocks_test == kode
    if idx.sum() < 10:
        print(f'Data {kode} tidak cukup di test set')
        return

    y_t  = y_true_inv[idx]
    y_lo = y_q10_inv[idx]
    y_md = y_q50_inv[idx]
    y_hi = y_q90_inv[idx]
    d    = dates_test[idx]

    # Ambil n_days terakhir
    y_t, y_lo, y_md, y_hi, d = y_t[-n_days:], y_lo[-n_days:], y_md[-n_days:], y_hi[-n_days:], d[-n_days:]

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.fill_between(d, y_lo, y_hi, alpha=0.2, color='#1565C0', label='Interval Q10-Q90 (80%)')
    ax.plot(d, y_md, color='#1565C0', linewidth=2, label='Prediksi Q50 (Median)', zorder=3)
    ax.plot(d, y_t,  color='#C62828', linewidth=1.5, label='Harga Aktual', zorder=4)
    ax.plot(d, y_lo, color='#1565C0', linewidth=0.8, linestyle='--', alpha=0.6)
    ax.plot(d, y_hi, color='#1565C0', linewidth=0.8, linestyle='--', alpha=0.6)

    for i in range(len(d)):
        color = '#4CAF50' if y_lo[i] <= y_t[i] <= y_hi[i] else '#F44336'
        ax.scatter(d[i], y_t[i], color=color, s=15, zorder=5)

    ax.set_title(f'Prediksi Interval Harga — {kode} ({n_days} hari terakhir test set)', fontsize=13, fontweight='bold')
    ax.set_ylabel('Harga (Rp)', fontsize=11)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'Rp {x:,.0f}'))
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(alpha=0.3)
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.savefig(f'viz_interval_{kode}.png', dpi=150, bbox_inches='tight')
    plt.show()

for kode in ['BBCA','BBRI','TLKM','ANTM']:
    if kode in unique_stocks_test:
        plot_prediction_interval(kode, n_days=60)
        print(f'✅ Visualisasi interval: {kode}')


---
## 9. Deteksi Arah Harga: Naik atau Turun?

Berdasarkan prediksi Q50 (median) dibandingkan dengan harga Close hari ini:
- Jika **Q50 > Close hari ini** → prediksi **NAIK** 📈
- Jika **Q50 < Close hari ini** → prediksi **TURUN** 📉

Range kenaikan  = Q90 − Close hari ini
Range penurunan = Close hari ini − Q10


In [ ]:
def predict_direction_and_range(kode):
    idx = stocks_test == kode
    if idx.sum() < 1:
        print(f'Data {kode} tidak tersedia di test set')
        return

    last = np.where(idx)[0][-1]
    close_today = close_test[last]
    q10 = y_q10_inv[last]
    q50 = y_q50_inv[last]
    q90 = y_q90_inv[last]
    tgl = pd.Timestamp(dates_test[last]).strftime('%d %b %Y')

    arah       = '📈 NAIK' if q50 > close_today else '📉 TURUN'
    range_naik = q90 - close_today
    range_trun = close_today - q10
    pct_naik   = (range_naik / close_today) * 100
    pct_turun  = (range_trun / close_today) * 100

    print(f'{'='*55}')
    print(f'  PREDIKSI INTERVAL HARGA — {kode}')
    print(f'{'='*55}')
    print(f'  Tanggal prediksi : {tgl}')
    print(f'  Harga Close (t)  : Rp {close_today:>10,.0f}')
    print(f'  Arah Prediksi    : {arah}')
    print(f'  Q10 (Batas Bawah): Rp {q10:>10,.0f}  (-Rp {range_trun:,.0f} / -{pct_turun:.2f}%)')
    print(f'  Q50 (Median)     : Rp {q50:>10,.0f}')
    print(f'  Q90 (Batas Atas) : Rp {q90:>10,.0f}  (+Rp {range_naik:,.0f} / +{pct_naik:.2f}%)')
    if q50 > close_today:
        print(f'  Range NAIK       : Rp {close_today:,.0f} → Rp {q90:,.0f} (+{pct_naik:.2f}%)')
    else:
        print(f'  Range TURUN      : Rp {q10:,.0f} → Rp {close_today:,.0f} (-{pct_turun:.2f}%)')
    print(f'{'='*55}
')

print('PREDIKSI INTERVAL & ARAH HARGA SEMUA SAHAM LQ45
')
for kode in sorted(unique_stocks_test):
    predict_direction_and_range(kode)


---
## 10. Ringkasan Hasil Evaluasi


In [ ]:
print("=" * 70)
print("   RINGKASAN EVALUASI MODEL LSTM + QUANTILE REGRESSION")
print("   Prediksi Interval Harga Saham LQ45")
print("=" * 70)
print("\n  Dataset            : IDX Stock Summary 2020-2024")
print(f"  Jumlah Saham       : {len(unique_stocks_test)} saham LQ45")
print(f"  Window Size        : {WINDOW_SIZE} hari")
print(f"  Training Samples   : {len(X_train):,}")
print(f"  Test Samples       : {len(X_test):,}")
print('  ' + '─'*65)
print(f"  METRIK EVALUASI                    NILAI")
print(f"  {'─'*65}")
print(f"  RMSE (Keseluruhan, Q50 vs Aktual) : {rmse_overall:>12,.2f}")
print(f"  RMSE Rata-rata per Saham          : {rmse_df['RMSE'].mean():>12,.2f}")
print(f"  RMSE Terbaik                      : {rmse_df['RMSE'].min():>12,.2f}  ({rmse_df['RMSE'].idxmin()})")
print(f"  RMSE Terburuk                     : {rmse_df['RMSE'].max():>12,.2f}  ({rmse_df['RMSE'].idxmax()})")
print(f"  {'─'*65}")
print(f"  Coverage Nominal (Q10–Q90)        : {'80.00%':>12}")
print(f"  Coverage Empiris                  : {cov_overall*100:>12.2f}%")
print(f"  Calibration Error (CE)            : {ce_overall*100:>12.4f}%")
status = '✅ Sangat Baik' if ce_overall < 0.05 else '✅ Baik' if ce_overall < 0.10 else '⚠️ Perlu Tuning'
print(f"  Status Kalibrasi                  : {status:>12}")
print(f"  {'─'*65}")
print("\n  Kuantil yang digunakan:")
print(f"    Q10 → Batas bawah interval (harga paling pesimis)")
print(f"    Q50 → Median / prediksi utama")
print(f"    Q90 → Batas atas interval (harga paling optimis)")
print("\n  Fungsi Loss : Pinball Loss (Quantile Loss)")
print(f"  Optimizer   : Adam (lr=0.001)")
print(f"  Arsitektur  : LSTM(128) → Dropout → LSTM(64) → Dense(32) → Output")
print("=" * 70)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PREDIKSI MANUAL — Input OHLCV harga sekarang (2026 atau kapanpun)
# Model belajar POLA PERUBAHAN dari data 2020-2024, lalu
# pola tersebut diaplikasikan ke harga input Anda sekarang.
# ══════════════════════════════════════════════════════════════════

def prediksi_saham(kode, open_price, high_price, low_price, close_price, volume):
    """
    Prediksi interval harga saham berdasarkan input OHLCV hari ini.
    Model menggunakan pola return historis → diaplikasikan ke harga input.
    Hasil: berapa rupiah naik/turun dari close_price yang Anda input.
    """
    if kode not in saham_list:
        print(f'❌ Saham {kode} tidak tersedia. Pilih dari: {saham_list}')
        return

    # Ambil 29 hari return historis terakhir dari dataset training
    hist = df_feat[df_feat['Stock_Code'] == kode][FEATURE_COLS].tail(WINDOW_SIZE - 1).values

    if len(hist) < WINDOW_SIZE - 1:
        print(f'❌ Data historis {kode} tidak cukup')
        return

    # Hitung return OHLCV hari input vs hari sebelumnya di dataset
    last_row = df_feat[df_feat['Stock_Code'] == kode][['Open','High','Low','Close','Volume']].iloc[-1]
    ret_O    = (open_price  - last_row['Open'])   / (last_row['Open']   + 1e-8)
    ret_H    = (high_price  - last_row['High'])   / (last_row['High']   + 1e-8)
    ret_L    = (low_price   - last_row['Low'])    / (last_row['Low']    + 1e-8)
    ret_C    = (close_price - last_row['Close'])  / (last_row['Close']  + 1e-8)

    # Volume relatif: z-score terhadap 20 hari terakhir
    vol_hist = df_feat[df_feat['Stock_Code'] == kode]['Volume'].tail(20)
    vol_rel  = (volume - vol_hist.mean()) / (vol_hist.std() + 1e-8)

    # Gabungkan: 29 hari historis + 1 hari input → sekuens 30 hari
    input_row = np.array([[ret_O, ret_H, ret_L, ret_C, vol_rel]])
    sequence  = np.vstack([hist, input_row])  # shape: (30, 5)
    X_input   = sequence.reshape(1, WINDOW_SIZE, len(FEATURE_COLS))

    # Prediksi return besok (Q10, Q50, Q90)
    ret_q10 = models[0.1].predict(X_input, verbose=0)[0][0]
    ret_q50 = models[0.5].predict(X_input, verbose=0)[0][0]
    ret_q90 = models[0.9].predict(X_input, verbose=0)[0][0]

    # Konversi return → harga absolut berdasarkan close_price INPUT
    q10 = close_price * (1 + ret_q10)
    q50 = close_price * (1 + ret_q50)
    q90 = close_price * (1 + ret_q90)

    # Hitung range naik/turun
    arah      = '📈 NAIK' if q50 > close_price else '📉 TURUN'
    range_up  = q90 - close_price
    range_dn  = close_price - q10
    pct_up    = (range_up  / close_price) * 100
    pct_dn    = (range_dn  / close_price) * 100
    chg_rp    = q50 - close_price
    chg_pct   = (chg_rp / close_price) * 100

    # Tampilkan hasil
    print(f'╔══════════════════════════════════════════════════════╗')
    print(f'  PREDIKSI INTERVAL HARGA — {kode}')
    print(f'╠══════════════════════════════════════════════════════╣')
    print(f'  INPUT HARI INI')
    print(f'  Open   : Rp {open_price:>10,.0f}')
    print(f'  High   : Rp {high_price:>10,.0f}')
    print(f'  Low    : Rp {low_price:>10,.0f}')
    print(f'  Close  : Rp {close_price:>10,.0f}')
    print(f'  Volume :    {volume:>10,.0f} lot')
    print(f'──────────────────────────────────────────────────────')
    print(f'  HASIL PREDIKSI BESOK')
    print(f'  Arah Prediksi    : {arah}')
    print(f'  Perubahan median : Rp {chg_rp:+,.0f}  ({chg_pct:+.2f}%)')
    print(f'──────────────────────────────────────────────────────')
    print(f'  Q10 (Batas Bawah): Rp {q10:>10,.0f}  (-Rp {range_dn:,.0f} / -{pct_dn:.2f}%)')
    print(f'  Q50 (Median)     : Rp {q50:>10,.0f}')
    print(f'  Q90 (Batas Atas) : Rp {q90:>10,.0f}  (+Rp {range_up:,.0f} / +{pct_up:.2f}%)')
    print(f'──────────────────────────────────────────────────────')
    if q50 > close_price:
        print(f'  Range NAIK : Rp {close_price:,.0f} → Rp {q90:,.0f}  (+{pct_up:.2f}%)')
    else:
        print(f'  Range TURUN: Rp {q10:,.0f} → Rp {close_price:,.0f}  (-{pct_dn:.2f}%)')
    print(f'╚══════════════════════════════════════════════════════╝
')


# ── CONTOH PENGGUNAAN ─────────────────────────────────────────────
# Ganti nilai OHLCV sesuai harga saham hari ini

prediksi_saham(
    kode        = 'BBCA',
    open_price  = 5900,
    high_price  = 5975,
    low_price   = 5825,
    close_price = 5900,
    volume      = 288919700
)

prediksi_saham(
    kode        = 'BBRI',
    open_price  = 4200,
    high_price  = 4280,
    low_price   = 4170,
    close_price = 4250,
    volume      = 95000000
)

prediksi_saham(
    kode        = 'TLKM',
    open_price  = 3800,
    high_price  = 3850,
    low_price   = 3770,
    close_price = 3820,
    volume      = 42000000
)


---
## Catatan untuk BAB 3 (Metodologi)

Berikut poin-poin yang bisa digunakan untuk menyusun BAB 3 Penelitian Ilmiah:

1. **Data** — IDX Stock Summary 2020–2024, 38 saham LQ45 valid (data ≥ 80%)
2. **Preprocessing** — Parsing tanggal, filter volume > 0, fix Open = 0, normalisasi Min-Max Scaling
3. **Fitur Input** — 5 fitur OHLCV: Open, High, Low, Close, Volume
4. **Target Output** — Harga Close hari berikutnya (t+1)
5. **Sliding Window** — Window size = 30 hari, Train:Test = 80%:20%
6. **Arsitektur Model** — LSTM(128) → Dropout(0.2) → LSTM(64) → Dropout(0.2) → BatchNorm → Dense(32) → Output(1)
7. **Quantile Regression** — 3 model terpisah untuk Q10, Q50, Q90 dengan Pinball Loss
8. **Evaluasi** — RMSE (akurasi prediksi median) + Calibration Error (kualitas interval)
